# Proyecto Final — Asistente de regalos
## Calidad de datos
Entendimiento y limpieza del dataset Online Retail II.

### 1. Carga de datos

In [67]:
import os
import pandas as pd

if not os.path.exists('../data/raw/online_retail_II_completo.csv'):
    hojas = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name=None)  # lento: solo corre la primera vez
    df = pd.concat(hojas.values(), ignore_index=True)
    df.to_csv('../data/raw/online_retail_II_completo.csv', index=False)
else:
    df = pd.read_csv('../data/raw/online_retail_II_completo.csv')  # rápido: ya está convertido

print(df.shape)

(1067371, 8)


**Hallazgo:** el dataset cargó con 1.067.371 filas y 8 columnas, 
coincide con lo documentado oficialmente para Online Retail II.

## Diccionario de variables

| Columna | Qué es | Notas |
|---|---|---|
| invoice_no | Número de factura/pedido | Si empieza con "C" es cancelación (1,8% de las filas) |
| stock_code | Código del producto | Algunos códigos no son productos reales: POST, DOT, M, BANK CHARGES, AMAZONFEE, B |
| description | Descripción del producto | Texto libre; 4.382 filas sin descripción (0,4%) |
| quantity | Cantidad comprada en esa línea | Puede ser negativa (cancelaciones o ajustes de depósito) |
| invoice_date | Fecha y hora de la transacción | Rango 2009-2011 |
| unit_price | Precio de una unidad | En libras esterlinas (GBP); puede venir en 0 (ajustes) |
| customer_id | Identificador del cliente | 243.007 filas sin cliente (22,8%) |
| country | País del cliente | 91,9% Reino Unido |

### 2. Exploración inicial
¿Cuántos datos faltan, qué tipos de dato hay, qué rango de fechas cubre?

In [68]:
df.info()
display(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 136.6 MB


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

**Hallazgo:** las columnas vinieron como `Invoice`, `Price` y 
`Customer ID` (no `InvoiceNo`, `UnitPrice`, `CustomerID`). Hay 
243.007 filas sin `Customer ID` (22,8%) y 4.382 sin `Description` 
(0,4%) — ambos se van a tratar en la limpieza.

### 3. Normalización de columnas
Las columnas se renombran para trabajar con un estándar fijo (snake_case).

In [69]:
df = df.rename(columns={
    'Invoice': 'invoice_no', 'StockCode': 'stock_code', 'Description': 'description',
    'Quantity': 'quantity', 'InvoiceDate': 'invoice_date', 'Price': 'unit_price',
    'Customer ID': 'customer_id', 'Country': 'country'
})
df.columns.tolist()

['invoice_no',
 'stock_code',
 'description',
 'quantity',
 'invoice_date',
 'unit_price',
 'customer_id',
 'country']

**Hallazgo:** las 8 columnas quedaron normalizadas en snake_case. 
A partir de acá, todo el código usa estos nombres.

### 4. Mirando la tabla y sus rangos

In [70]:
display(df.head())

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [71]:
# Nuevo
display(df[df['invoice_no'] == '489434'])# filtro TODAS las filas de esta factura, sin que head() las corte

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom


#Nuevo **Hallazgo:** la factura 489434 tiene 8 líneas (no 5, como parecía con 
head()). De esas, 7 son múltiplo de 6 y 1 no (cantidad 10) — o sea, 
87,5% del patrón bulk, no el 100% que se había dicho antes. También 
confirma que códigos con letra al final (79323P, 79323W) pueden ser 
productos reales (variantes de color), no solo ajustes administrativos.

### 5. Rangos y valores extremos

In [72]:
display(df.describe())

,quantity,unit_price,customer_id
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


**Hallazgo:** quantity tiene un rango de -80.995 a +80.995, mientras 
que unit_price llega a -53.594,36. Son valores extremos poco 
habituales — hay que investigarlos antes de decidir cómo tratarlos.

### 6. Casos sospechosos: cantidad o precio no válidos

In [73]:
display(df[(df['quantity'] <= 0) | (df['unit_price'] <= 0)].head(10))
print('cantidad <= 0:', (df['quantity'] <= 0).sum())
print('unit_price <= 0:', (df['unit_price'] <= 0).sum())

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


cantidad <= 0: 22950
unit_price <= 0: 6207


**Hallazgo:** 22.950 filas tienen quantity <= 0, más que las 19.494 
cancelaciones formales (invoice_no con "C") — hay ~3.456 filas con 
cantidad negativa o cero que no están marcadas como cancelación 
oficial (se investiga en la sección 9). El filtro quantity > 0 las 
saca a todas de todos modos. Aparte, 6.207 filas tienen 
unit_price <= 0 (incluye los 4 casos de "Adjust bad debt" de la 
sección 7, más ceros).

### 6.1 Duplicados exactos

In [74]:
#Nuevo
print(df.duplicated().sum(), "filas duplicadas exactas")  # cuenta filas idénticas en TODAS sus columnas

34335 filas duplicadas exactas


In [75]:
#Nuevo
display(df[df.duplicated(keep=False)].sort_values(['invoice_no', 'stock_code']).head(10))  # ejemplo real, no solo el número

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


#Nuevo **Hallazgo:** 34.335 filas duplicadas exactas (3,2% del dataset). 
Un ejemplo real: "VINTAGE SNAKES & LADDERS" aparece 3 veces en la 
misma factura, cada una con quantity=1 — no se ve como 1 compra 
de 3 unidades en una sola línea, sino como 3 líneas separadas.

### 6.2 ¿Se repite el patrón en los 34.335?

In [76]:
#Nuevo
duplicados = df[df.duplicated(keep=False)]
print(duplicados['quantity'].value_counts().head())  # ¿la mayoría tiene quantity=1, como en el ejemplo?

repeticiones = duplicados.groupby(['invoice_no', 'stock_code']).size()
print(repeticiones.describe())  # cuántas veces se repite cada combinación factura+producto, en promedio

quantity
1     31263
2      9661
12     4896
6      4300
3      3776
Name: count, dtype: int64
count    32597.000000
mean         2.062828
std          0.377345
min          2.000000
25%          2.000000
50%          2.000000
75%          2.000000
max         20.000000
dtype: float64


#Nuevo **Hallazgo:** el 75% de las combinaciones factura+producto 
duplicadas se repite exactamente 2 veces (mediana = 2). Si 
descartáramos duplicados quedándonos con 1 sola fila, perderíamos 
la cantidad real que se llevó el cliente (ej. quedaría quantity=1 
en vez de 3).

**Decisión:** no se usa drop_duplicates() a secas. Se agrupan las 
filas duplicadas y se suma quantity, para conservar la cantidad 
real que se llevó el cliente en vez de perderla al quedarse con 
una sola fila.

### 7. Investigando el precio negativo extremo

In [77]:
display(df[df['unit_price'] < 0][['stock_code', 'description', 'unit_price']].drop_duplicates())

,stock_code,description,unit_price
179403,B,Adjust bad debt,-53594.36
276274,B,Adjust bad debt,-44031.79
403472,B,Adjust bad debt,-38925.87
825444,B,Adjust bad debt,-11062.06


**Hallazgo:** stock_code "B" = "Adjust bad debt" — ajuste contable 
por deuda incobrable, no una venta real. Se suma a la lista de 
códigos no-producto a sacar en la limpieza (junto con POST, DOT, 
M, BANK CHARGES, AMAZONFEE).

### 7.1 Auditoría de códigos que no siguen el patrón estándar

In [78]:
#Nuevo
codigos_raros = df[~df['stock_code'].str.match(r'^\d{5}$')]['stock_code'].unique()  # códigos que NO son 5 dígitos seguidos
print(len(codigos_raros), "códigos distintos no son 5 dígitos seguidos")  # cuántos hay en total
print(sorted(codigos_raros)[:30])  # primeros 30, para no inundar la pantalla

1715 códigos distintos no son 5 dígitos seguidos
['10002R', '10123C', '10123G', '10124A', '10124C', '10124G', '15044A', '15044B', '15044C', '15044D', '15056BL', '15056N', '15056P', '15056bl', '15056n', '15056p', '15058A', '15058B', '15058C', '15059A', '15060A', '15060B', '15060b', '16020C', '16065A', '16151A', '16155H', '16156A', '16156L', '16156N']


#Nuevo **Hallazgo:** 1.715 códigos no siguen el patrón de 5 dígitos. La mayoría son 
variantes de color/tamaño de un producto base (ej. 10124A, 10124C, 
10124G) — productos reales, no administrativos.

### 7.2 ¿Son duplicados por mayúscula/minúscula?

In [79]:
#Nuevo
print(df['stock_code'].nunique(), "códigos distintos (mayúscula y minúscula por separado)")  # cuenta cada variante de caso como distinta
print(df['stock_code'].str.upper().nunique(), "códigos distintos después de unificar mayúsculas")  # unifico y cuento de nuevo

5305 códigos distintos (mayúscula y minúscula por separado)
5132 códigos distintos después de unificar mayúsculas


#Nuevo **Hallazgo:** 5.305 códigos distintos bajan a 5.132 al unificar mayúsculas — 173 
eran el mismo producto escrito distinto (ej. 15056BL y 15056bl). 
Confirma que normalizar con .str.upper() en la limpieza es necesario, 
no una precaución de más.

### 7.3 Acotar los candidatos reales a revisar

In [80]:
#Nuevo
precio_cero_siempre = df.groupby('stock_code')['unit_price'].apply(lambda s: (s == 0).all())  # True si TODAS las ventas de ese código tienen precio 0
sospechosos = precio_cero_siempre[precio_cero_siempre].index.tolist()  # me quedo solo con los códigos marcados True
print(len(sospechosos), "códigos con TODAS sus filas en precio 0 (candidatos reales a revisar)")

373 códigos con TODAS sus filas en precio 0 (candidatos reales a revisar)


#Nuevo **Hallazgo:** 373 códigos tienen precio 0 en el 100% de sus ventas. La mayoría son 
productos reales de 5 dígitos (muestras o promociones) — el filtro 
unit_price > 0 ya los saca a todos sin necesitar código nuevo.

### 7.4 Revisar los más raros entre los sospechosos

In [81]:
#Nuevo
sospechosos_especiales = ['C3', 'GIFT', 'gift_0001_60', 'gift_0001_90', 'DCGS0006', 'DCGS0016', 'DCGS0027',
                           'DCGS0036', 'DCGS0039', 'DCGS0053', 'DCGS0055', 'DCGS0056', 'DCGS0057', 'DCGS0059',
                           'DCGS0060', 'DCGS0066P', 'DCGS0067', 'DCGS0071', 'DCGS0073', 'DCGS0074',
                           'DCGSLBOY', 'DCGSLGIRL']
display(df[df['stock_code'].isin(sospechosos_especiales)][['stock_code', 'description']].drop_duplicates())  # miro sus descripciones reales

,stock_code,description
75138,gift_0001_60,NaN
75141,gift_0001_90,NaN
80337,DCGS0006,NaN
80338,DCGS0016,NaN
80339,DCGS0027,NaN
80342,DCGS0036,NaN
80361,DCGS0039,NaN
80953,DCGS0060,NaN
81143,DCGS0056,NaN
82846,DCGS0059,NaN


#Nuevo **Hallazgo:** de los 373 códigos con precio siempre en 0, 22 (los 
"gift_", "GIFT", "DCGS0XXX", "C3") tienen descripción vacía o solo 
"ebay" — sin evidencia de ser productos identificables.

**Decisión:** no se agregan a la lista de códigos no-producto. 
catalogo_productos() ya descarta cualquier stock_code sin 
descripción, así que estos 22 nunca llegan a mostrarse ni a 
recomendarse en el asistente.

### 8. País y cancelaciones

In [82]:
display(df['country'].value_counts().head(10))
print(df['invoice_no'].astype(str).str.startswith('C').sum(), "filas de cancelación")

country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64

19494 filas de cancelación


**Hallazgo:** United Kingdom es el 91,9% de las filas (981.330 de 
1.067.371). Hay 19.494 filas de cancelación (1,8%).

### 9. Cantidad negativa SIN ser cancelación formal

In [83]:
mask = (df['quantity'] < 0) & (~df['invoice_no'].astype(str).str.startswith('C'))  # cantidad negativa Y NO es cancelación formal
print(mask.sum(), "filas con cantidad negativa que no son cancelación 'C'")
display(df[mask].head(10))

3457 filas con cantidad negativa que no son cancelación 'C'


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom


In [84]:
print('todas con unit_price == 0:', (df[mask]['unit_price'] == 0).all())  # ¿se cumple en TODAS, o solo en la muestra que vi?
print('todas con customer_id nulo:', df[mask]['customer_id'].isna().all())

todas con unit_price == 0: True
todas con customer_id nulo: True


**Hallazgo:** las 3.457 filas con cantidad negativa que no son 
cancelación formal tienen el 100% unit_price = 0 y 100% 
customer_id nulo — confirmado sobre las 3.457 filas completas, no 
solo sobre la muestra de 10. Son ajustes de inventario escritos a 
mano ("lost", "damages", "short", "invcd as..."), no ventas ni 
cancelaciones de cliente. El filtro unit_price > 0 (ya planeado) 
las saca a todas — no hace falta un filtro nuevo para esto.

### 9.1 Reconciliando la diferencia de 1 fila (3.456 vs 3.457)

In [85]:
#Nuevo 
mask_rara = df['invoice_no'].astype(str).str.startswith('C') & (df['quantity'] >= 0)  # facturas "C" con cantidad NO negativa
print(mask_rara.sum(), "facturas 'C' con cantidad >= 0")  # si da más de 0, ahí está la fila que faltaba
display(df[mask_rara])  # para ver de qué se trata, si aparece alguna

1 facturas 'C' con cantidad >= 0


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
76799,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,NaN,United Kingdom


#Nuevo **Hallazgo:** la diferencia de 1 fila (3.457 vs 3.456) era una 
factura "C" con stock_code "M" (Manual) y cantidad positiva — un 
ajuste administrativo cargado con prefijo de cancelación, no una 
cancelación real de cliente.

#Nuevo **Decisión:** no hace falta ningún cambio en el código de limpieza 
— esta fila ya se excluye sola con el filtro de códigos no-producto 
existente (stock_code "M" está en CODIGOS_NO_PRODUCTO).

### 10. ¿El patrón de la primera factura se repite en todo el dataset?

In [86]:
#Nuevo
proporcion_multiplo = (df['quantity'] % 6 == 0).groupby(df['invoice_no']).mean()  # % de líneas múltiplo de 6, por factura
display(proporcion_multiplo.describe())

n_productos = df.groupby('invoice_no')['stock_code'].nunique()  # productos distintos por factura
es_bulk = (proporcion_multiplo >= 0.8) & (n_productos >= 5)  # 80%+ en múltiplos de 6 Y al menos 5 productos
print(f"% facturas bulk (proporción 80% + mínimo 5 productos): {es_bulk.mean():.1%}")

count    53628.000000
mean         0.352582
std          0.353950
min          0.000000
25%          0.000000
50%          0.250000
75%          0.647059
max          1.000000
Name: quantity, dtype: float64

% facturas bulk (proporción 80% + mínimo 5 productos): 7.1%


#Nuevo **Hallazgo:** la proporción de líneas en múltiplos de 6 por factura 
tiene mediana 0,25 y percentil 75 de 0,647 — la mayoría de las 
facturas no se acerca al patrón bulk. Con el criterio corregido 
(80%+ de las líneas en múltiplos de 6, Y al menos 5 productos 
distintos), el 7,1% de las facturas califica como bulk. Este 
criterio reemplaza al anterior (100% exacto, sin mínimo de 
productos), que había dado 10,1% con una regla más frágil.

### 11. Impacto de conservar filas sin customer_id

In [87]:
#Nuevo
con_cliente = df.dropna(subset=['customer_id'])  # como quedaría si exigimos cliente

print(f"Facturas totales: {df['invoice_no'].nunique()} | con cliente: {con_cliente['invoice_no'].nunique()}")  # cuántas facturas enteras se pierden
print(f"Productos distintos totales: {df['stock_code'].nunique()} | con cliente: {con_cliente['stock_code'].nunique()}")  # si además se achica la variedad de productos

Facturas totales: 53628 | con cliente: 44876
Productos distintos totales: 5305 | con cliente: 4646


#Nuevo **Hallazgo:** exigir customer_id elimina 8.752 facturas completas 
(16,3%) y 659 productos distintos (12,4% del catálogo) que solo 
aparecen en compras sin cliente registrado.

**Decisión:** se arman 2 datasets a partir de aquí, como había 
propuesto Mafe. Para RFM y segmentación mayorista/minorista: 
pipeline_completo(conservar_sin_cliente=False) — ahí SÍ hace 
falta saber quién compró. Para el modelo de recomendación 
(matriz factura-producto): pipeline_completo(conservar_sin_cliente=True) 
— el modelo no pregunta quién compró, y descartar esas filas le 
esconde 1 de cada 8 productos del catálogo.

## Conclusión: qué aprendimos antes de limpiar

El dataset tiene 1.067.371 filas y 8 columnas, consistente con lo 
documentado oficialmente. Las columnas no venían estandarizadas, 
así que las normalizamos a snake_case.

**Valores extremos investigados — 2 hipótesis razonables, no 
confirmadas con certeza:**
1. Stock_code "B" trae la descripción "Adjust bad debt". Nuestra 
   interpretación: es un ajuste contable por deuda incobrable, no 
   una venta — apoyada en que el precio es negativo y enorme, pero 
   no hay forma de confirmarlo con quien armó el dataset.
2. 3.457 filas con cantidad negativa, sin ser cancelación formal, 
   traen descripciones como "lost", "damages", "short", con 100% 
   precio=0 y 100% sin cliente. Nuestra lectura: pérdida o daño de 
   inventario, no una compra — razonable dado el patrón, pero 
   también una interpretación, no un hecho verificado.

Reino Unido concentra el 91,9% de las filas (hecho medido); 
cancelaciones formales son el 1,8% (hecho medido).

**Códigos de producto:** 1.715 no siguen el patrón de 5 dígitos 
(hecho medido). Revisando sus descripciones, la mayoría parecen 
variantes de color o tamaño de un mismo producto (confirmado 
viendo el texto real, ej. 79323P/79323W = "PINK/WHITE CHERRY 
LIGHTS"). Unificar mayúsculas bajó los códigos distintos de 5.305 
a 5.132 (hecho medido). De 373 códigos con precio siempre en 0, 
22 no tienen descripción real — ya quedan afuera del catálogo 
automáticamente.

**Duplicados exactos:** 34.335 filas (3,2%, hecho medido). Se 
agrupa y suma `quantity` en vez de descartar, para no perder 
cantidad real comprada.

**Patrón de "facturas bulk":** le llamamos así a una hipótesis de 
reposición al por mayor, inferida de que las cantidades vienen en 
múltiplos de 6 y hay varios productos distintos — no es un dato 
confirmado, es un patrón que medimos (7,1% de las facturas, con 
80%+ de líneas en múltiplos de 6 y mínimo 5 productos) y al que le 
dimos una interpretación de negocio razonable, apoyada en que el 
propio dataset documenta que "muchos clientes son mayoristas".

**Clientes sin identificar:** exigir customer_id perdería 8.752 
facturas (16,3%) y 659 productos (12,4% del catálogo) — esto sí es 
un hecho medido directo. Por eso se generan 2 datasets: uno exige 
cliente (RFM), otro lo conserva (modelo de recomendación).

Con esto, la limpieza queda justificada con evidencia propia — 
distinguiendo qué se midió directamente y qué es una interpretación 
razonable de esa evidencia, no un hecho confirmado al 100%.

#Nuevo ### 12. Aplicando la limpieza real

Corremos el pipeline de limpieza (src/data/clean.py) para generar 
los datasets finales.

In [99]:
import sys
sys.path.append('..')

from src.data import clean
import importlib
importlib.reload(clean)  # por si el módulo ya estaba cargado en memoria con una versión vieja
from src.data.clean import pipeline_completo

**Hallazgo:** la primera corrida del pipeline mostró 3 pasos 
consecutivos con el mismo número de filas ("sin duplicados", "sin 
descripción vacía", "con customer_id") — señal de que groupby() 
estaba descartando de más. Por default, pandas tira las filas con 
NaN en las columnas de agrupamiento, y en ese punto del pipeline 
customer_id y description todavía tenían NaN.

**Decisión:** se agregó dropna=False al groupby() de limpiar_datos() 
en clean.py, para que agrupe los NaN entre sí en vez de descartarlos. 
Con el fix, cada filtro saca solo lo que le corresponde.

In [105]:
df_rfm, log_rfm = pipeline_completo('../data/raw/online_retail_II.xlsx', 
                                     '../data/processed/online_retail_rfm.parquet')
display(log_rfm)

                               paso   filas  facturas
                            inicial 1067371     53628
                  sin cancelaciones 1047877     45336
              cantidad y precio > 0 1041670     40077
            sin códigos no-producto 1037427     39580
sin duplicados (cantidades sumadas)  993249     39580
              sin descripción vacía  993249     39580
                    con customer_id  766521     36650


,paso,filas,facturas
0,inicial,1067371,53628
1,sin cancelaciones,1047877,45336
2,cantidad y precio > 0,1041670,40077
3,sin códigos no-producto,1037427,39580
4,sin duplicados (cantidades sumadas),993249,39580
5,sin descripción vacía,993249,39580
6,con customer_id,766521,36650


**Hallazgo:** dataset para RFM: 766.521 filas, 36.650 facturas. 
Se pierden 2.930 facturas al exigir customer_id (7,4% de las 
39.580 que quedaban en ese punto del pipeline).

In [104]:
import importlib
importlib.reload(clean)
from src.data.clean import pipeline_completo

df_modelo, log_modelo = pipeline_completo('../data/raw/online_retail_II.xlsx',
                                            '../data/processed/online_retail_modelo.parquet',
                                            conservar_sin_cliente=True)
display(log_modelo)

                               paso   filas  facturas
                            inicial 1067371     53628
                  sin cancelaciones 1047877     45336
              cantidad y precio > 0 1041670     40077
            sin códigos no-producto 1037427     39580
sin duplicados (cantidades sumadas)  993249     39580
              sin descripción vacía  993249     39580


,paso,filas,facturas
0,inicial,1067371,53628
1,sin cancelaciones,1047877,45336
2,cantidad y precio > 0,1041670,40077
3,sin códigos no-producto,1037427,39580
4,sin duplicados (cantidades sumadas),993249,39580
5,sin descripción vacía,993249,39580


**Hallazgo:** el dataset del modelo (conservar_sin_cliente=True) 
tiene 993.249 filas y 39.580 facturas — 2.930 facturas más que el 
de RFM (766.521 filas, 36.650 facturas), confirmando que se 
recuperan las ventas sin cliente identificado para el modelo de 
recomendación, sin afectar el dataset usado para RFM.